In [ ]:
import pandas as pd
file_path = "D:/100k.csv"
df = pd.read_csv(file_path)
df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from itertools import combinations
from sklearn.model_selection import train_test_split

pos_tags = ['AUX', 'CCONJ', 'INTJ', 'PRON', 'VERB', 'SCONJ', 'ADJ', 'NOUN', 'ADV', 'PROPN', 'ADP']
pos_columns = [f'pos_{i}' for i in range(1, 32)]

results = []

for size in range(1, 12):
    print(f"\n--- Subset Size: {size} ---")
    for combo in combinations(pos_tags, size):
        combination = set(combo)
        filtered_rows = []

        for _, row in df.iterrows():
            tags = row[pos_columns].tolist()
            filtered = [tag if tag in combination else np.nan for tag in tags]
            non_nan = [tag for tag in filtered if pd.notna(tag)]
            filtered_rows.append(non_nan)

        max_len = max(len(row) for row in filtered_rows)
        if max_len == 0:
            continue

        shifted_df = pd.DataFrame([
            row + [np.nan] * (max_len - len(row)) for row in filtered_rows
        ], columns=[f'pos_{i+1}' for i in range(max_len)])

        shifted_df['Sentiment'] = df['Sentiment'].values

        # Encode
        X_full = shifted_df.drop(columns=['Sentiment']).astype(str)
        y = shifted_df['Sentiment']
        label_encoders = {}
        for col in X_full.columns:
            le = LabelEncoder()
            X_full[col] = le.fit_transform(X_full[col])
            label_encoders[col] = le

        # Train/test split and model
        X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
        model = RandomForestClassifier()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        print(f"Combo: {combo} -> Accuracy: {acc:.4f}")
        results.append({'combo': combo, 'accuracy': acc})

# Top 10 overall
top_10 = sorted(results, key=lambda x: x['accuracy'], reverse=True)[:10]
print("\nTop 10 Combinations:")
for i, res in enumerate(top_10, 1):
    print(f"{i}. Tags: {res['combo']} -> Accuracy: {res['accuracy']:.4f} ")

# Best per size
best_per_size = {}
for res in results:
    size = len(res['combo'])
    if size not in best_per_size or res['accuracy'] > best_per_size[size]['accuracy']:
        best_per_size[size] = res

print("\nBest combination per subset size:")
for size in sorted(best_per_size):
    res = best_per_size[size]
    print(f"Size {size}: Tags {res['combo']} -> Accuracy: {res['accuracy']:.4f} ")

